# Allen–Cahn: referencia DeepXDE v1.12.2

Esta notebook reproduce el ejemplo oficial con backend PyTorch y una GPU Kaggle T4x2. Ejecuta cinco seeds en secuencia. El límite de 7.200 s empieza en la primera celda de código e incluye pip, descarga y preparación. Un watchdog reserva tiempo para guardar resultados. Si alguna fase se corta, la seed y la corrida quedan como truncadas y no reciben puntuación completa.

El `.mat` oficial del tag v1.12.2 contiene `t` con forma `(1, 101)` y `u` con forma `(101, 201)`, sin una variable `x`; el ejemplo upstream intenta leer `x`. Se conserva `u` sin cambios y se reconstruye solo el eje espacial como 201 puntos uniformes en `[-1, 1]`, según el dominio del mismo ejemplo. URL, tag, SHA-256, formas y conversión quedan en `provenance.json`.

Antes de ejecutar, configura Kaggle con Internet habilitado y acelerador **GPU T4x2**. DeepXDE usa `cuda:0`; la segunda T4 se adjunta y se registra, pero el FNN estándar no distribuye automáticamente el trabajo entre ambas.


In [ ]:
# El reloj comienza antes de instalar dependencias o descargar el campo.
import csv
import datetime as dt
import hashlib
import importlib.metadata as package_metadata
import importlib.util
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import urllib.request

RUN_STARTED_MONO = time.monotonic()
RUN_STARTED_UTC = dt.datetime.now(dt.timezone.utc).isoformat()
GLOBAL_BUDGET_SECONDS = 7200
FINALIZE_RESERVE_SECONDS = 180
GLOBAL_DEADLINE = RUN_STARTED_MONO + GLOBAL_BUDGET_SECONDS
TRAIN_DEADLINE = GLOBAL_DEADLINE - FINALIZE_RESERVE_SECONDS
SEEDS = [0, 1, 2, 3, 4]
OUTPUT_DIR = Path('/kaggle/working/allen_cahn_deepxde_reference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_FIELDS = [
    'seed', 'status', 'recipe_complete', 'adam_steps', 'lbfgs_completed',
    'field_l2_relative', 'mean_abs_residual', 'ic_max_abs_error',
    'bc_max_abs_error', 'energy_initial', 'energy_final', 'energy_delta',
    'energy_mean', 'elapsed_seconds', 'active_gpu_seconds',
    'allocated_gpu_seconds', 'allocated_gpu_hours', 'cost_usd_estimate',
    'cost_basis', 'stop_reason',
]

def blank_seed(seed):
    return {
        'seed': seed,
        'status': 'not_started',
        'recipe_complete': False,
        'adam_steps': 0,
        'lbfgs_completed': False,
        'field_l2_relative': None,
        'mean_abs_residual': None,
        'ic_max_abs_error': None,
        'bc_max_abs_error': None,
        'energy_initial': None,
        'energy_final': None,
        'energy_delta': None,
        'energy_mean': None,
        'elapsed_seconds': 0.0,
        'active_gpu_seconds': 0.0,
        'allocated_gpu_seconds': 0.0,
        'allocated_gpu_hours': 0.0,
        'cost_usd_estimate': None,
        'cost_basis': 'Kaggle does not expose monetary billing; GPU time is recorded.',
        'stop_reason': None,
    }

RESULTS = {
    'schema_version': '1.0',
    'benchmark': 'DeepXDE Allen-Cahn v1.12.2 reference',
    'run_started_utc': RUN_STARTED_UTC,
    'global_budget_seconds': GLOBAL_BUDGET_SECONDS,
    'setup_shared_seconds': None,
    'overall_status': 'running',
    'score_valid': False,
    'seeds': [blank_seed(seed) for seed in SEEDS],
}
PROVENANCE = {
    'benchmark': 'DeepXDE Allen-Cahn example',
    'deepxde_version_requested': '1.12.2',
    'source_tag': 'v1.12.2',
    'source_commit_short': '414e457',
    'source_url': 'https://github.com/lululxvi/deepxde/blob/v1.12.2/examples/pinn_forward/Allen_Cahn.py',
    'dataset_url': 'https://raw.githubusercontent.com/lululxvi/deepxde/v1.12.2/examples/dataset/Allen_Cahn.mat',
    'dataset_tag': 'v1.12.2',
    'dataset_sha256_expected': 'f9175685675685ab8d2a850f0d212d16c709f55ed89b8fe078493ddcd3724aa0',
    'dataset_git_blob_sha1': '2617d4be2746ea1689e04b9d29f0217ac1899a5f',
    'dataset_sha256_observed': None,
    'dataset_raw_shapes': None,
    'dataset_u_shape': None,
    'dataset_x_reconstruction': None,
    'runtime': {},
    'setup_status': 'pending',
    'setup_error': None,
    'notes': [
        'The official v1.12.2 Allen_Cahn.mat has t (1, 101) and u (101, 201), but no x variable.',
        'u is used unchanged; x is reconstructed from the upstream Interval(-1, 1) and u.shape[1].',
        'DeepXDE runs on cuda:0; Kaggle T4x2 is required, but this reference FNN does not use data parallelism.',
        'The 7200-second clock starts in the first code cell; Kaggle queue and kernel provisioning happen before Python starts and are not observable here.',
    ],
}

def update_overall_status():
    complete = len(RESULTS['seeds']) == 5 and all(
        row['status'] == 'complete' and row['recipe_complete'] for row in RESULTS['seeds']
    )
    RESULTS['score_valid'] = bool(complete)
    if complete:
        RESULTS['overall_status'] = 'complete'
    elif any(row['status'] in ('truncated', 'truncated_evaluation', 'running') for row in RESULTS['seeds']):
        RESULTS['overall_status'] = 'truncated'
    elif any(row['status'] == 'not_started' for row in RESULTS['seeds']):
        RESULTS['overall_status'] = 'incomplete'
    elif all(row['status'].endswith('_blocked') for row in RESULTS['seeds']):
        RESULTS['overall_status'] = 'blocked'
    else:
        RESULTS['overall_status'] = 'incomplete'

def persist_outputs():
    RESULTS['run_finished_utc'] = dt.datetime.now(dt.timezone.utc).isoformat()
    RESULTS['total_wall_seconds'] = round(time.monotonic() - RUN_STARTED_MONO, 3)
    update_overall_status()
    (OUTPUT_DIR / 'results.json').write_text(json.dumps(RESULTS, indent=2, allow_nan=False), encoding='utf-8')
    (OUTPUT_DIR / 'provenance.json').write_text(json.dumps(PROVENANCE, indent=2, allow_nan=False), encoding='utf-8')
    with (OUTPUT_DIR / 'results.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(RESULTS['seeds'])

persist_outputs()
print(f'Budget: {GLOBAL_BUDGET_SECONDS}s; output directory: {OUTPUT_DIR}')


In [ ]:
# Instala únicamente si falta una dependencia o si DeepXDE no coincide con el tag fijado.
os.environ['DDE_BACKEND'] = 'pytorch'
SETUP_OK = False
SETUP_ERROR = None
DDE = None
TORCH = None
SCIPY_LOADMAT = None
GPU_NAMES = []
GPU_COUNT = 0

try:
    missing = []
    try:
        installed_dde = package_metadata.version('deepxde')
    except package_metadata.PackageNotFoundError:
        installed_dde = None
    if installed_dde != '1.12.2':
        missing.append('deepxde==1.12.2')
    if importlib.util.find_spec('scipy') is None:
        missing.append('scipy')
    if missing:
        remaining = int(TRAIN_DEADLINE - time.monotonic())
        if remaining <= 0:
            raise TimeoutError('Se agotó el presupuesto antes de instalar dependencias.')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing],
            check=True,
            timeout=remaining,
        )
    import deepxde as DDE
    import torch as TORCH
    from scipy.io import loadmat as SCIPY_LOADMAT

    if DDE.__version__ != '1.12.2':
        raise RuntimeError(f'DeepXDE requerido 1.12.2; instalado {DDE.__version__}.')
    if not TORCH.cuda.is_available():
        raise RuntimeError('CUDA no está disponible; selecciona GPU T4x2 en Kaggle.')
    GPU_COUNT = TORCH.cuda.device_count()
    GPU_NAMES = [TORCH.cuda.get_device_name(i) for i in range(GPU_COUNT)]
    if GPU_COUNT != 2 or not all('T4' in name.upper() for name in GPU_NAMES):
        raise RuntimeError(f'Se requiere T4x2; se detectó count={GPU_COUNT}, devices={GPU_NAMES}.')
    TORCH.cuda.set_device(0)
    DDE.config.set_random_seed(0)
    SETUP_OK = True
    PROVENANCE['setup_status'] = 'ready'
    PROVENANCE['runtime'] = {
        'python': sys.version,
        'deepxde': DDE.__version__,
        'torch': TORCH.__version__,
        'scipy': package_metadata.version('scipy'),
        'numpy': package_metadata.version('numpy'),
        'cuda_available': TORCH.cuda.is_available(),
        'cuda_device_count': GPU_COUNT,
        'cuda_devices': GPU_NAMES,
        'training_device': 'cuda:0',
        'backend': os.environ['DDE_BACKEND'],
        'default_float': 'DeepXDE default (float32)',
    }
except Exception as exc:
    SETUP_ERROR = f'{type(exc).__name__}: {exc}'
    PROVENANCE['setup_status'] = 'dependency_blocked' if 'DeepXDE' in SETUP_ERROR or 'deepxde' in SETUP_ERROR.lower() or 'scipy' in SETUP_ERROR.lower() else 'environment_blocked'
    PROVENANCE['setup_error'] = SETUP_ERROR
    for row in RESULTS['seeds']:
        row['status'] = PROVENANCE['setup_status']
        row['stop_reason'] = SETUP_ERROR

RESULTS['setup_shared_seconds'] = round(time.monotonic() - RUN_STARTED_MONO, 3)
persist_outputs()
print(f'Setup: {PROVENANCE["setup_status"]}; {PROVENANCE.get("setup_error") or PROVENANCE["runtime"]}')


In [ ]:
# Descarga y fija la referencia por SHA-256 antes de iniciar cualquier seed.
DATA_URL = PROVENANCE['dataset_url']
EXPECTED_SHA256 = PROVENANCE['dataset_sha256_expected']
MAT_PATH = OUTPUT_DIR / 'Allen_Cahn.mat'
REFERENCE_OK = False
REFERENCE_ERROR = None
X_REF = None
T_REF = None
U_REF = None
X_FIELD = None
Y_FIELD = None

if SETUP_OK:
    try:
        remaining = int(TRAIN_DEADLINE - time.monotonic())
        if remaining <= 0:
            raise TimeoutError('No queda presupuesto para obtener la referencia.')
        request = urllib.request.Request(DATA_URL, headers={'User-Agent': 'allen-cahn-deepxde-kaggle-reference/1.0'})
        with urllib.request.urlopen(request, timeout=min(60, remaining)) as response, MAT_PATH.open('wb') as target:
            shutil.copyfileobj(response, target)
        observed_sha256 = hashlib.sha256(MAT_PATH.read_bytes()).hexdigest()
        PROVENANCE['dataset_sha256_observed'] = observed_sha256
        if observed_sha256 != EXPECTED_SHA256:
            raise RuntimeError(f'SHA-256 de Allen_Cahn.mat no coincide: {observed_sha256}')

        raw = SCIPY_LOADMAT(MAT_PATH)
        raw_shapes = {name: list(value.shape) for name, value in raw.items() if not name.startswith('__') and hasattr(value, 'shape')}
        if 't' not in raw or 'u' not in raw:
            raise KeyError(f'Faltan t/u en el dataset oficial. Variables: {sorted(raw_shapes)}')
        T_REF = raw['t'].reshape(-1).astype('float64')
        U_REF = raw['u'].astype('float64')
        if U_REF.ndim != 2 or U_REF.shape[0] != T_REF.size:
            raise ValueError(f'Formas no compatibles: t={T_REF.shape}, u={U_REF.shape}')
        if 'x' in raw:
            X_REF = raw['x'].reshape(-1).astype('float64')
            x_reconstruction = 'No aplica: x estaba presente en el .mat.'
        else:
            # El propio ejemplo v1.12.2 fija Interval(-1, 1); el ancho de u fija nx.
            X_REF = __import__('numpy').linspace(-1.0, 1.0, U_REF.shape[1], dtype='float64')
            x_reconstruction = 'linspace(-1, 1, u.shape[1]); u se conserva sin cambios ni interpolación.'
        if X_REF.size != U_REF.shape[1]:
            raise ValueError(f'x tiene {X_REF.size} puntos, u espera {U_REF.shape[1]}.')
        if not (math.isclose(float(X_REF[0]), -1.0) and math.isclose(float(X_REF[-1]), 1.0)):
            raise ValueError(f'El eje x no cubre [-1, 1]: {X_REF[[0, -1]]}.')
        if not (math.isclose(float(T_REF[0]), 0.0) and math.isclose(float(T_REF[-1]), 1.0)):
            raise ValueError(f'El eje t no cubre [0, 1]: {T_REF[[0, -1]]}.')
        PROVENANCE['dataset_raw_shapes'] = raw_shapes
        PROVENANCE['dataset_u_shape'] = list(U_REF.shape)
        PROVENANCE['dataset_x_reconstruction'] = x_reconstruction
        import numpy as np
        xx, tt = np.meshgrid(X_REF, T_REF)
        X_FIELD = np.column_stack((xx.ravel(), tt.ravel())).astype(np.float32)
        Y_FIELD = U_REF.reshape(-1, 1).astype(np.float32)
        if X_FIELD.shape[0] != Y_FIELD.shape[0]:
            raise ValueError(f'Raster mismatch: X={X_FIELD.shape}, u={Y_FIELD.shape}.')
        REFERENCE_OK = True
        PROVENANCE['reference_status'] = 'ready'
    except Exception as exc:
        REFERENCE_ERROR = f'{type(exc).__name__}: {exc}'
        PROVENANCE['reference_status'] = 'dataset_blocked'
        PROVENANCE['reference_error'] = REFERENCE_ERROR
        for row in RESULTS['seeds']:
            row['status'] = 'dataset_blocked'
            row['stop_reason'] = REFERENCE_ERROR
else:
    PROVENANCE['reference_status'] = 'not_attempted'
    PROVENANCE['reference_error'] = SETUP_ERROR

persist_outputs()
if REFERENCE_OK:
    print(f"Reference ready: sha256={PROVENANCE['dataset_sha256_observed']}; u={U_REF.shape}; x={X_REF.size}; t={T_REF.size}")
else:
    print(f'Reference unavailable: {REFERENCE_ERROR or SETUP_ERROR}')


In [ ]:
# Código de una seed: receta upstream y métricas del campo, residual, condiciones y energía.
TRAINER_SOURCE = r'''import argparse, datetime, hashlib, json, os, sys, time
os.environ['DDE_BACKEND'] = 'pytorch'
import numpy as np
import torch
import deepxde as dde
from scipy.io import loadmat

parser = argparse.ArgumentParser()
parser.add_argument('--seed', type=int, required=True)
parser.add_argument('--mat', required=True)
parser.add_argument('--expected-sha256', required=True)
parser.add_argument('--deadline', type=float, required=True)
parser.add_argument('--progress', required=True)
parser.add_argument('--result', required=True)
parser.add_argument('--gpu-count', type=int, required=True)
args = parser.parse_args()
started = time.monotonic()

def atomic_json(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def base_result():
    return {
        'seed': args.seed, 'status': 'running', 'recipe_complete': False,
        'adam_steps': 0, 'lbfgs_completed': False,
        'field_l2_relative': None, 'mean_abs_residual': None,
        'ic_max_abs_error': None, 'bc_max_abs_error': None,
        'energy_initial': None, 'energy_final': None, 'energy_delta': None,
        'energy_mean': None, 'elapsed_seconds': 0.0,
        'active_gpu_seconds': 0.0, 'allocated_gpu_seconds': 0.0,
        'allocated_gpu_hours': 0.0, 'cost_usd_estimate': None,
        'cost_basis': 'Kaggle does not expose monetary billing; GPU time is recorded.',
        'stop_reason': None,
    }

result = base_result()
def finish(status, reason=None):
    elapsed = time.monotonic() - started
    result['status'] = status
    result['stop_reason'] = reason
    result['elapsed_seconds'] = round(elapsed, 3)
    result['active_gpu_seconds'] = round(elapsed, 3)
    result['allocated_gpu_seconds'] = round(elapsed * args.gpu_count, 3)
    result['allocated_gpu_hours'] = round(elapsed * args.gpu_count / 3600.0, 6)
    atomic_json(args.result, result)
    atomic_json(args.progress, result)

try:
    mat_sha = hashlib.sha256(open(args.mat, 'rb').read()).hexdigest()
    if mat_sha != args.expected_sha256:
        raise RuntimeError('Dataset SHA-256 changed between setup and worker.')
    raw = loadmat(args.mat)
    t = raw['t'].reshape(-1).astype(np.float64)
    u_true = raw['u'].astype(np.float64)
    if 'x' in raw:
        x = raw['x'].reshape(-1).astype(np.float64)
    else:
        x = np.linspace(-1.0, 1.0, u_true.shape[1], dtype=np.float64)
    if u_true.shape != (t.size, x.size):
        raise ValueError(f'Incompatible reference shapes: t={t.shape}, x={x.shape}, u={u_true.shape}')

    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    dde.config.set_random_seed(args.seed)
    torch.cuda.set_device(0)

    geom = dde.geometry.Interval(-1, 1)
    timedomain = dde.geometry.TimeDomain(0, 1)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)
    d = 0.001

    def pde(inputs, y):
        dy_t = dde.grad.jacobian(y, inputs, i=0, j=1)
        dy_xx = dde.grad.hessian(y, inputs, i=0, j=0)
        return dy_t - d * dy_xx - 5 * (y - y**3)

    def output_transform(inputs, y):
        xcoord = inputs[:, 0:1]
        tcoord = inputs[:, 1:2]
        return xcoord**2 * torch.cos(np.pi * xcoord) + tcoord * (1 - xcoord**2) * y

    class Watchdog(dde.callbacks.Callback):
        def __init__(self, phase):
            super().__init__()
            self.phase = phase
            self.triggered = False
            self.last_write = 0
        def on_epoch_end(self):
            now = time.monotonic()
            state = self.model.train_state
            step = int(getattr(state, 'step', 0))
            if now - self.last_write >= 20:
                result['phase'] = self.phase
                result['phase_step'] = step
                result['elapsed_seconds'] = round(now - started, 3)
                atomic_json(args.progress, result)
                self.last_write = now
            if now >= args.deadline:
                self.triggered = True
                self.model.stop_training = True

    data = dde.data.TimePDE(
        geomtime, pde, [], num_domain=8000, num_boundary=400, num_initial=800
    )
    net = dde.nn.FNN([2] + [20] * 3 + [1], 'tanh', 'Glorot normal')
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile('adam', lr=1e-3)
    adam_watchdog = Watchdog('adam')
    adam_step_before = int(getattr(model.train_state, 'step', 0))
    model.train(iterations=40000, callbacks=[adam_watchdog], display_every=10000)
    adam_steps = max(0, int(getattr(model.train_state, 'step', 0)) - adam_step_before)
    result['adam_steps'] = adam_steps
    result['training'] = {'adam_requested': 40000, 'adam_completed': adam_steps, 'lbfgs_requested': True, 'lbfgs_completed': False}
    if adam_watchdog.triggered or adam_steps < 40000:
        finish('truncated', 'Global watchdog stopped Adam before 40000 steps.')
        sys.exit(0)

    lbfgs_watchdog = Watchdog('lbfgs')
    model.compile('L-BFGS')
    model.train(callbacks=[lbfgs_watchdog], display_every=1000)
    if lbfgs_watchdog.triggered:
        finish('truncated', 'Global watchdog stopped L-BFGS before completion.')
        sys.exit(0)
    result['lbfgs_completed'] = True
    result['training']['lbfgs_completed'] = True

    # Conserva margen para cerrar archivos y consolidar las cinco filas.
    if time.monotonic() + 90 >= args.deadline:
        finish('truncated_evaluation', 'Entrenamiento terminó, pero no quedó margen para completar la evaluación.')
        sys.exit(0)

    xx, tt = np.meshgrid(x, t)
    X = np.column_stack((xx.ravel(), tt.ravel())).astype(np.float32)
    y_flat = u_true.reshape(-1, 1).astype(np.float32)
    pred_parts, residual_parts, energy_parts = [], [], []

    def energy_density(inputs, y):
        ux = dde.grad.jacobian(y, inputs, i=0, j=0)
        return 0.5 * d * ux**2 + 1.25 * (y**2 - 1.0)**2

    for start in range(0, len(X), 2048):
        if time.monotonic() + 30 >= args.deadline:
            finish('truncated_evaluation', 'Watchdog durante la evaluación del campo.')
            sys.exit(0)
        batch = X[start:start + 2048]
        pred_parts.append(model.predict(batch))
        residual_parts.append(model.predict(batch, operator=pde))
        energy_parts.append(model.predict(batch, operator=energy_density))

    y_pred = np.vstack(pred_parts)
    residual = np.vstack(residual_parts)
    energy_density_grid = np.vstack(energy_parts).reshape(u_true.shape)
    field_l2 = float(dde.metrics.l2_relative_error(y_flat, y_pred))
    mean_abs_residual = float(np.mean(np.abs(residual)))

    x_ic = np.column_stack((x, np.zeros_like(x))).astype(np.float32)
    ic_pred = model.predict(x_ic).reshape(-1)
    ic_target = x**2 * np.cos(np.pi * x)
    ic_error = float(np.max(np.abs(ic_pred - ic_target)))
    t_bc = t.astype(np.float32)
    left = np.column_stack((np.full_like(t_bc, -1.0), t_bc))
    right = np.column_stack((np.full_like(t_bc, 1.0), t_bc))
    bc_pred = np.vstack((model.predict(left), model.predict(right))).reshape(-1)
    bc_error = float(np.max(np.abs(bc_pred + 1.0)))

    dx = np.diff(x)
    energy_by_time = np.sum(
        0.5 * (energy_density_grid[:, :-1] + energy_density_grid[:, 1:]) * dx[None, :],
        axis=1,
    )
    result.update({
        'field_l2_relative': field_l2,
        'mean_abs_residual': mean_abs_residual,
        'ic_max_abs_error': ic_error,
        'bc_max_abs_error': bc_error,
        'energy_initial': float(energy_by_time[0]),
        'energy_final': float(energy_by_time[-1]),
        'energy_delta': float(energy_by_time[-1] - energy_by_time[0]),
        'energy_mean': float(np.mean(energy_by_time)),
        'energy_definition': 'integral_x [0.5*d*u_x^2 + 1.25*(u^2-1)^2] dx; trapezoid on official x raster',
        'metrics_grid_shape': list(u_true.shape),
        'recipe_complete': True,
        'training': {
            'adam_requested': 40000,
            'adam_completed': adam_steps,
            'lbfgs_requested': True,
            'lbfgs_completed': True,
        },
        'stop_reason': None,
    })
    finish('complete')
except SystemExit:
    raise
except Exception as exc:
    finish('worker_failed', f'{type(exc).__name__}: {exc}')
    raise
'''

if not SETUP_OK or not REFERENCE_OK:
    print('Seeds remain blocked; no score is valid.')
else:
    WORKER_PATH = OUTPUT_DIR / 'train_seed.py'
    WORKER_PATH.write_text(TRAINER_SOURCE, encoding='utf-8')
    setup_shared_seconds = round(time.monotonic() - RUN_STARTED_MONO, 3)
    RESULTS['setup_shared_seconds'] = setup_shared_seconds
    PROVENANCE['setup_shared_seconds'] = setup_shared_seconds
    PROVENANCE['training'] = {
        'adam_steps': 40000,
        'adam_optimizer': 'Adam',
        'adam_learning_rate': 0.001,
        'lbfgs_optimizer': 'DeepXDE L-BFGS defaults for v1.12.2',
        'samples': {'domain': 8000, 'boundary': 400, 'initial': 800},
        'architecture': [2, 20, 20, 20, 1],
        'activation': 'tanh',
        'initializer': 'Glorot normal',
        'hard_output_transform': 'x^2*cos(pi*x) + t*(1-x^2)*NN(x,t)',
        'seeds': SEEDS,
    }
    PROVENANCE['reference_metrics'] = {
        'field_l2_relative': 'DeepXDE l2_relative_error over all official u grid points',
        'residual': 'mean(abs(u_t - 0.001*u_xx - 5*(u-u^3))) over the same grid',
        'initial_condition': 'max abs error against x^2*cos(pi*x) on the official x raster at t=0',
        'boundary_condition': 'max abs error against -1 at x=-1,+1 on the official t raster',
        'energy': 'integral_x [0.5*d*u_x^2 + 1.25*(u^2-1)^2] dx; report initial/final/delta/mean',
    }
    PROVENANCE['cost_accounting'] = {
        'active_gpu_seconds': 'wall seconds on cuda:0',
        'allocated_gpu_seconds': 'wall seconds times two attached T4 devices',
        'allocated_gpu_hours': 'allocated_gpu_seconds / 3600',
        'cost_usd_estimate': 'null unless KAGGLE_T4X2_USD_PER_HOUR is explicitly set',
    }
    persist_outputs()

    for row in RESULTS['seeds']:
        if time.monotonic() >= TRAIN_DEADLINE:
            row['status'] = 'truncated'
            row['stop_reason'] = 'Global 7200-second budget leaves only the finalization reserve.'
            persist_outputs()
            continue
        seed = row['seed']
        progress_path = OUTPUT_DIR / f'seed_{seed}_progress.json'
        result_path = OUTPUT_DIR / f'seed_{seed}_result.json'
        log_path = OUTPUT_DIR / f'seed_{seed}.log'
        row['status'] = 'running'
        row['stop_reason'] = None
        persist_outputs()
        worker_args = [
            sys.executable, str(WORKER_PATH),
            '--seed', str(seed),
            '--mat', str(MAT_PATH),
            '--expected-sha256', EXPECTED_SHA256,
            '--deadline', str(TRAIN_DEADLINE),
            '--progress', str(progress_path),
            '--result', str(result_path),
            '--gpu-count', str(GPU_COUNT),
        ]
        seed_started = time.monotonic()
        with log_path.open('w', encoding='utf-8') as log_handle:
            proc = subprocess.Popen(worker_args, stdout=log_handle, stderr=subprocess.STDOUT)
            while proc.poll() is None and time.monotonic() < TRAIN_DEADLINE:
                time.sleep(min(2.0, max(0.1, TRAIN_DEADLINE - time.monotonic())))
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
                row['status'] = 'truncated'
                row['stop_reason'] = 'Parent watchdog terminated the seed at the global training deadline.'
                if progress_path.exists():
                    try:
                        progress = json.loads(progress_path.read_text(encoding='utf-8'))
                        row.update({k: v for k, v in progress.items() if k != 'seed'})
                        row['status'] = 'truncated'
                        row['recipe_complete'] = False
                        row['stop_reason'] = 'Parent watchdog terminated the seed at the global training deadline.'
                    except Exception:
                        pass
            elif proc.returncode == 0 and result_path.exists():
                try:
                    row.update(json.loads(result_path.read_text(encoding='utf-8')))
                except Exception as exc:
                    row['status'] = 'worker_failed'
                    row['stop_reason'] = f'Could not read worker result: {type(exc).__name__}: {exc}'
            else:
                row['status'] = 'worker_failed'
                try:
                    tail = log_path.read_text(encoding='utf-8', errors='replace')[-4000:]
                except Exception:
                    tail = ''
                row['stop_reason'] = f'Worker exit code {proc.returncode}; log tail: {tail}'

        wall = round(time.monotonic() - seed_started, 3)
        row['elapsed_seconds'] = wall
        row['active_gpu_seconds'] = wall
        row['allocated_gpu_seconds'] = round(wall * GPU_COUNT, 3)
        row['allocated_gpu_hours'] = round(wall * GPU_COUNT / 3600.0, 6)
        price_text = os.environ.get('KAGGLE_T4X2_USD_PER_HOUR')
        if price_text:
            try:
                row['cost_usd_estimate'] = round(float(price_text) * row['allocated_gpu_hours'], 6)
                row['cost_basis'] = 'allocated_gpu_hours * KAGGLE_T4X2_USD_PER_HOUR'
            except ValueError:
                row['cost_usd_estimate'] = None
                row['cost_basis'] = 'Invalid KAGGLE_T4X2_USD_PER_HOUR; monetary cost unavailable.'
        persist_outputs()
        if row['status'] in ('truncated', 'truncated_evaluation'):
            for later in RESULTS['seeds']:
                if later['seed'] > seed and later['status'] == 'not_started':
                    later['status'] = 'truncated'
                    later['stop_reason'] = 'Earlier seed consumed the remaining global budget.'
            persist_outputs()
            break

    PROVENANCE['notebook_finish_utc'] = dt.datetime.now(dt.timezone.utc).isoformat()
    persist_outputs()
    print(json.dumps({'overall_status': RESULTS['overall_status'], 'score_valid': RESULTS['score_valid'], 'results': str(OUTPUT_DIR / 'results.json'), 'csv': str(OUTPUT_DIR / 'results.csv')}, indent=2))


## Archivos de salida

- `results.json`: estado global, métricas por seed, tiempos y motivo de truncamiento.
- `results.csv`: una fila por seed con L2 del campo, residual, IC/BC, energía y coste/tiempo.
- `provenance.json`: tag/URL/hash/formas, reconstrucción del eje `x`, versiones de runtime y receta.
- `seed_<n>.log` y archivos JSON de progreso: trazas del watchdog por seed.

Una corrida solo tiene `score_valid: true` cuando las cinco seeds completan los 40.000 pasos Adam, L-BFGS y todas las métricas. La estimación monetaria queda nula si Kaggle no proporciona un precio explícito; se reporta el consumo de GPU asignada en horas.
